# Image Geolocation — CLIP Embeddings + Geocell Classification

Predicts the (latitude, longitude) a street-level/landscape photo was taken at,
plus a confidence radius in km — using a **frozen CLIP image encoder**
and a small **trained classification head** over precomputed geographic cells
(geocells). No reverse image search, no mapping APIs, fully offline inference.

## Folder layout this notebook expects

- `reference/` — our already-trained, validated artifacts. **Never overwritten by
  this notebook.** Contains `head.pt`, `config.json`, `geocell_summary.csv`, and
  `administrative_maps/` (Natural Earth boundaries — read directly from here
  whenever geocells are built, never copied into `data/`).
- `data/` — everything else, all of it disposable/regeneratable. `data/sample/`
  is the only part committed to git (a small demo set); every other `data/`
  subfolder is created automatically by the first cell below and gets seeded
  from `reference/` where relevant.

## Quick facts

- Section 4 auto-detects your data: uses `data/images/` if it has images,
  otherwise falls back to the small `data/sample/` demo set — clone and run
  top-to-bottom with no setup, in under a minute, either way.
- To reproduce the numbers printed in this notebook (median error, coverage,
  country accuracy), populate `data/images/` with the full dataset, or point
  Section 4's `CUSTOM_DATA_PATH`/`CUSTOM_LABELS_CSV` at a Kaggle-mounted
  dataset — see Section 4 for exactly where to make that change.
- Bring your own dataset the same way: any folder of images + a CSV with
  columns `image_id, latitude, longitude` works.


## Optional: environment setup

Uncomment whichever of these apply to where you're running this.


In [ ]:
# If running on Kaggle/Colab and this repo isn't already present:
# !git clone https://github.com/Divyant-Jayakumar/GeoGuesser
# %cd GeoGuesser

# If geopandas/shapely/transformers aren't preinstalled (common on fresh Colab):
# !pip install -q geopandas shapely transformers


## 1. Setup & config

Seeds, device selection, the config dict, and the folder structure. Every
`data/` subfolder is created here if it doesn't exist yet — nothing below this
cell should ever fail with a missing-directory error.


In [ ]:
import os
import json
import random
import shutil

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ---- Config ----
CONFIG = {
    "backbone": "openai/clip-vit-large-patch14-336",
    "embedding_dim": 768,
    "num_geocells": 2005,          # overwritten once geocells are built/loaded below
    "min_points_for_geocells": 10000,
    "min_points_for_geocell_merge": 30,
    "max_points_per_geocell": 150,
}

# DATA_PATH / LABELS_CSV are chosen in Section 4 (Data loading), not here --
# that's the cell that actually reads them.

# ---- Folder structure ----
DATA_DIRS = [
    "data/sample/images",
    "data/images",
    "data/custom_images",
    "data/splits",
    "data/geocells",
    "data/embeddings",
    "data/checkpoints",
    "data/predictions",
    "assets",
    "results",
]
for d in DATA_DIRS:
    os.makedirs(d, exist_ok=True)

# ---- Seed working copies from the immutable reference/ artifacts ----
def seed_from_reference(reference_path, working_path):
    if os.path.exists(working_path):
        return
    if not os.path.exists(reference_path):
        return
    if os.path.isdir(reference_path):
        shutil.copytree(reference_path, working_path)
    else:
        os.makedirs(os.path.dirname(working_path), exist_ok=True)
        shutil.copy(reference_path, working_path)

seed_from_reference("reference/geocell_summary.csv", "data/geocells/geocell_summary.csv")
seed_from_reference("reference/head.pt", "data/checkpoints/head.pt")
seed_from_reference("reference/config.json", "data/checkpoints/config.json")

# Admin boundaries are static reference data -- always read directly from
# reference/, never copied into data/. There's nothing dataset-specific
# about them, so there's no working copy to keep in sync or get stale.
ADMIN0_PATH = "reference/administrative_maps/ne_10m_admin_0_countries.geojson"
ADMIN1_PATH = "reference/administrative_maps/ne_10m_admin_1_states_provinces.geojson"
WORKING_GEOCELLS_PATH = "data/geocells/geocell_summary.csv"
REFERENCE_GEOCELLS_PATH = "reference/geocell_summary.csv"

print("Setup complete.")


## 2. Model & utility definitions

The `Head` module, haversine distance helpers, CLIP loading/embedding helpers,
and the `predict()` function used by both quick inference (below) and
evaluation (later). Defined once, up front, since quick inference needs all of
this before any training happens.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from transformers import CLIPModel, CLIPProcessor


class Head(nn.Module):
    # Small trainable MLP on top of a frozen CLIP embedding. Predicts a
    # distribution over geocells, plus two scalars (radius_a, radius_b) that
    # convert the distribution's probability-weighted spread into a raw
    # confidence radius in km.
    def __init__(self, embedding_dim, num_geocells, hidden_dim=512):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.cls_head = nn.Linear(hidden_dim, num_geocells)
        self.radius_a = nn.Parameter(torch.tensor(1.0))
        self.radius_b = nn.Parameter(torch.tensor(50.0))

    def forward(self, embeddings):
        h = self.trunk(embeddings)
        return self.cls_head(h)


def haversine_km(lat1, lon1, lat2, lon2):
    # numpy version — used for inference-time and geocell-building distance math
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def haversine_torch(lat1, lon1, lat2, lon2):
    # torch version — used inside the training loop so gradients can flow
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(torch.deg2rad, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = torch.sin(dlat / 2) ** 2 + torch.cos(lat1) * torch.cos(lat2) * torch.sin(dlon / 2) ** 2
    return 2 * R * torch.arcsin(torch.sqrt(a.clamp(min=0, max=1)))


def load_clip_backbone(model_name, device):
    model = CLIPModel.from_pretrained(model_name).to(device)
    processor = CLIPProcessor.from_pretrained(model_name)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model, processor


def get_clip_embedding(image_path, model, processor, device):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.get_image_features(**inputs)
    # some transformers versions return a wrapped output object here instead
    # of a raw tensor -- handle both defensively rather than assuming one
    embedding = output.pooler_output if hasattr(output, "pooler_output") else output
    embedding = embedding / embedding.norm(dim=-1, keepdim=True)
    return embedding.squeeze(0).cpu().numpy()


def compute_spread_km(probs, centroids, pred_lat, pred_lon):
    # probability-weighted average distance from the predicted centroid to
    # every geocell centroid — the model's own uncertainty signal.
    # Deliberately recomputed here rather than looked up from a cached
    # geocell-geocell distance matrix: this is the one row of that matrix
    # inference actually needs, and it's cheap (O(num_geocells)) to derive
    # on the spot from centroids alone -- no separate distance data has to
    # be generated or stored for a new scheme, or kept in reference/ for
    # the existing one.
    dists = haversine_km(pred_lat, pred_lon, centroids[:, 0], centroids[:, 1])
    return float(np.average(dists, weights=probs))


def radius_from_spread(spread_km, radius_a, radius_b):
    return radius_a * spread_km + radius_b


def predict(image_path, head_weights_path, config_path, geocell_csv_path,
            clip_model, clip_processor, device):
    with open(config_path) as f:
        cfg = json.load(f)

    geocells = pd.read_csv(geocell_csv_path).sort_values("geocell_id").reset_index(drop=True)
    centroids = geocells[["centroid_lat", "centroid_lon"]].values

    head_local = Head(embedding_dim=cfg["embedding_dim"], num_geocells=cfg["num_geocells"]).to(device)
    head_local.load_state_dict(torch.load(head_weights_path, map_location=device))
    head_local.eval()

    embedding = get_clip_embedding(image_path, clip_model, clip_processor, device)
    embedding_t = torch.tensor(embedding, dtype=torch.float32, device=device).unsqueeze(0)

    with torch.no_grad():
        logits = head_local(embedding_t)
        probs = F.softmax(logits, dim=-1).squeeze(0).cpu().numpy()

    best_idx = int(np.argmax(probs))
    pred_lat = float(centroids[best_idx, 0])
    pred_lon = float(centroids[best_idx, 1])

    spread_km = compute_spread_km(probs, centroids, pred_lat, pred_lon)
    radius_km = radius_from_spread(spread_km, head_local.radius_a.item(), head_local.radius_b.item())

    return pred_lat, pred_lon, radius_km


print("Model and utility definitions loaded.")


## 3. Quick inference (uses `reference/`, not `data/`)

Runs our already-trained model on every image in `data/sample/` in a few
seconds, comparing each prediction against the known ground-truth coordinate
(since the sample set has labels) so the error is visible immediately. This
always reads from `reference/` — it is unaffected by anything a retrain
further down writes into `data/checkpoints/`. Results are also saved to
`data/predictions/quick_inference_results.csv`.


In [ ]:
SAMPLE_IMAGES_DIR = "data/sample/images"
SAMPLE_LABELS_CSV = "data/sample/images.csv"
QUICK_INFERENCE_OUTPUT_CSV = "data/predictions/quick_inference_results.csv"

missing_reference = [
    p for p in ["reference/head.pt", "reference/config.json", "reference/geocell_summary.csv"]
    if not os.path.exists(p)
]

if missing_reference:
    print(f"Missing required reference/ file(s): {missing_reference}")
    print("Add your trained artifacts to reference/ before running this cell.")
elif not os.path.exists(SAMPLE_LABELS_CSV):
    print(f"{SAMPLE_LABELS_CSV} not found -- add sample images and a matching images.csv first.")
else:
    sample_labels = pd.read_csv(SAMPLE_LABELS_CSV)
    if len(sample_labels) == 0:
        print(f"{SAMPLE_LABELS_CSV} has no rows -- add at least one sample image and re-run this cell.")
    else:
        clip_model, clip_processor = load_clip_backbone(CONFIG["backbone"], DEVICE)

        results = []
        print(f"{'image_id':40s} {'pred (lat, lon)':24s} {'radius':>9s}   {'true (lat, lon)':24s} {'error':>10s}")
        for _, row in sample_labels.iterrows():
            image_path = os.path.join(SAMPLE_IMAGES_DIR, row["image_id"])

            pred_lat, pred_lon, pred_radius_km = predict(
                image_path=image_path,
                head_weights_path="reference/head.pt",
                config_path="reference/config.json",
                geocell_csv_path="reference/geocell_summary.csv",
                clip_model=clip_model,
                clip_processor=clip_processor,
                device=DEVICE,
            )
            error_km = haversine_km(pred_lat, pred_lon, row["latitude"], row["longitude"])

            pred_str = f"({pred_lat:.4f}, {pred_lon:.4f})"
            true_str = f"({row['latitude']:.4f}, {row['longitude']:.4f})"
            print(f"{row['image_id']:40s} {pred_str:24s} {pred_radius_km:7.1f}km   {true_str:24s} {error_km:8.1f}km")

            results.append({
                "image_id": row["image_id"],
                "pred_lat": pred_lat,
                "pred_lon": pred_lon,
                "pred_radius_km": pred_radius_km,
                "true_lat": row["latitude"],
                "true_lon": row["longitude"],
                "error_km": error_km,
            })

        os.makedirs(os.path.dirname(QUICK_INFERENCE_OUTPUT_CSV), exist_ok=True)
        pd.DataFrame(results).to_csv(QUICK_INFERENCE_OUTPUT_CSV, index=False)
        print(f"Saved results to {QUICK_INFERENCE_OUTPUT_CSV}")

# To try a single photo of your own instead of the sample set, see Section 11.


## 4. Data loading & train/val split

**This is the cell that controls which dataset Sections 4–10 train and
evaluate on.** By default it uses the full dataset in `data/images/`; if that
folder is empty, it automatically falls back to the small `data/sample/` demo
set and prints a message saying so. See the `CUSTOM_DATA_PATH` /
`CUSTOM_LABELS_CSV` comments in the cell below to point this at a Kaggle
dataset or anywhere else instead.

Also creates (or reuses) a deterministic train/val split, kept as its own
file rather than a column on `images.csv` so bringing your own dataset never
requires adding a split column to your labels file.


In [ ]:
# ---- Choose your dataset here ----
# To use a Kaggle-mounted dataset (or any other custom location), set BOTH
# of these to real paths -- the images folder AND the labels CSV:
#   CUSTOM_DATA_PATH  = "/kaggle/input/<your-dataset-slug>/images"
#   CUSTOM_LABELS_CSV = "/kaggle/input/<your-dataset-slug>/images.csv"
# The labels CSV must have columns: image_id, latitude, longitude.
# Leave both as None to auto-detect instead (see below).
CUSTOM_DATA_PATH = None
CUSTOM_LABELS_CSV = None

FULL_DATA_PATH = "data/images"
FULL_LABELS_CSV = "data/images.csv"
SAMPLE_DATA_PATH = "data/sample/images"
SAMPLE_LABELS_CSV = "data/sample/images.csv"


def _folder_has_images(folder):
    if not os.path.isdir(folder):
        return False
    return any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in os.listdir(folder))


if CUSTOM_DATA_PATH and CUSTOM_LABELS_CSV:
    DATA_PATH = CUSTOM_DATA_PATH
    LABELS_CSV = CUSTOM_LABELS_CSV
    print(f"Using custom dataset: {DATA_PATH}")
elif _folder_has_images(FULL_DATA_PATH):
    DATA_PATH = FULL_DATA_PATH
    LABELS_CSV = FULL_LABELS_CSV
    print(f"Using the full dataset from {DATA_PATH}")
else:
    DATA_PATH = SAMPLE_DATA_PATH
    LABELS_CSV = SAMPLE_LABELS_CSV
    print(f"{FULL_DATA_PATH}/ is empty -- falling back to the small demo set at {DATA_PATH}")

if not os.path.exists(LABELS_CSV):
    if _folder_has_images(DATA_PATH):
        raise FileNotFoundError(
            f"Found images in {DATA_PATH}/ but no labels file at {LABELS_CSV} -- "
            f"create a matching CSV (columns: image_id, latitude, longitude) before running this notebook."
        )
    raise FileNotFoundError(
        f"{LABELS_CSV} not found -- point LABELS_CSV at a real file before running this notebook."
    )

images_df = pd.read_csv(LABELS_CSV)
if len(images_df) == 0:
    raise ValueError(
        f"{LABELS_CSV} has no rows -- add image labels before running the rest of this notebook."
    )
print(f"Loaded {len(images_df)} labeled images from {LABELS_CSV}")

SPLIT_CSV = "data/splits/train_val_split.csv"

if os.path.exists(SPLIT_CSV):
    split_df = pd.read_csv(SPLIT_CSV)
    print("Loaded existing train/val split")
else:
    from sklearn.model_selection import train_test_split
    train_ids, val_ids = train_test_split(
        images_df["image_id"], test_size=0.2, random_state=SEED
    )
    split_df = pd.DataFrame({
        "image_id": pd.concat([train_ids, val_ids]),
        "split": ["train"] * len(train_ids) + ["val"] * len(val_ids),
    })
    split_df.to_csv(SPLIT_CSV, index=False)
    print("Created new train/val split")

images_df = images_df.merge(split_df, on="image_id")
print(images_df["split"].value_counts())


## 5. Geocell creation

Builds geographic cells from Natural Earth admin-0 (country) + admin-1
(state/province) boundaries: each point is assigned to a country/admin-1 unit,
small units are merged with their neighbors within the same country until each
cell has enough points, and any resulting cell that's too large is split with
OPTICS density clustering. Noise points from OPTICS, and points needing a
sub-cell assignment, are matched to their nearest cluster centroid — this is
the practical equivalent of Voronoi tessellation for our purposes (a
classification target + a centroid), without needing to construct and clip
actual polygons.

Only runs when there's enough data to build a sensible scheme
(`min_points_for_geocells`). Below that, this cell falls back to
`reference/geocell_summary.csv` and just assigns each point to its nearest
existing cell — this is what keeps the small demo dataset working.


In [ ]:
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import OPTICS


def split_oversized_cell(group, min_cell_size):
    coords = group[["latitude", "longitude"]].values
    clustering = OPTICS(min_samples=min_cell_size).fit(coords)
    labels = clustering.labels_.copy()

    valid_mask = labels != -1
    if valid_mask.sum() == 0:
        return np.zeros(len(group), dtype=int)

    centroids = (
        pd.DataFrame(coords[valid_mask], columns=["latitude", "longitude"])
        .assign(label=labels[valid_mask])
        .groupby("label")[["latitude", "longitude"]]
        .mean()
    )

    noise_idx = np.where(labels == -1)[0]
    for i in noise_idx:
        lat, lon = coords[i]
        dists = haversine_km(lat, lon, centroids["latitude"].values, centroids["longitude"].values)
        labels[i] = centroids.index[np.argmin(dists)]

    return labels


def build_geocells(df, admin0_path, admin1_path, min_cell_size, max_cell_size):
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=[Point(lon, lat) for lat, lon in zip(df["latitude"], df["longitude"])],
        crs="EPSG:4326",
    )

    admin0 = gpd.read_file(admin0_path)[["ADMIN", "geometry"]].rename(columns={"ADMIN": "country"})
    admin1 = gpd.read_file(admin1_path)[["name", "geometry"]].rename(columns={"name": "admin1"})

    gdf = gpd.sjoin(gdf, admin0, how="left", predicate="intersects").drop(columns="index_right")
    gdf = gpd.sjoin(gdf, admin1, how="left", predicate="intersects").drop(columns="index_right")
    gdf["country"] = gdf["country"].fillna("unknown")
    gdf["admin1"] = gdf["admin1"].fillna("unknown")

    gdf["geocell_key"] = gdf["country"].astype(str) + " / " + gdf["admin1"].astype(str)

    # merge small admin1 units within the same country until each bucket has >= min_cell_size
    merged_keys = {}
    for country, group in gdf.groupby("country"):
        counts = group["geocell_key"].value_counts().sort_values()
        bucket, bucket_count = [], 0
        for key, cnt in counts.items():
            bucket.append(key)
            bucket_count += cnt
            if bucket_count >= min_cell_size:
                merged_name = f"{country}__cell_{len(merged_keys)}"
                for k in bucket:
                    merged_keys[k] = merged_name
                bucket, bucket_count = [], 0
        if bucket:
            fallback = next((v for k, v in merged_keys.items() if k in counts.index), None)
            target = fallback if fallback is not None else f"{country}__cell_0"
            for k in bucket:
                merged_keys[k] = target

    gdf["geocell_key"] = gdf["geocell_key"].map(merged_keys)

    # split any oversized merged cell via OPTICS + nearest-centroid assignment
    final_labels = pd.Series(index=gdf.index, dtype="int64")
    next_id = 0
    for key, group in gdf.groupby("geocell_key"):
        if len(group) <= max_cell_size:
            final_labels.loc[group.index] = next_id
            next_id += 1
        else:
            sub_labels = split_oversized_cell(group, min_cell_size)
            for sub_label in np.unique(sub_labels):
                mask = sub_labels == sub_label
                final_labels.loc[group.index[mask]] = next_id
                next_id += 1

    gdf["geocell_id"] = final_labels.values

    geocell_summary_local = (
        gdf.groupby("geocell_id")
        .agg(
            country=("country", "first"),
            num_points=("geocell_id", "count"),
            centroid_lat=("latitude", "mean"),
            centroid_lon=("longitude", "mean"),
        )
        .reset_index()
        .sort_values("geocell_id")
        .reset_index(drop=True)
    )

    return gdf[["image_id", "geocell_id", "country"]], geocell_summary_local


def assign_to_nearest_geocell(lat, lon, summary):
    dists = haversine_km(lat, lon, summary["centroid_lat"].values, summary["centroid_lon"].values)
    return summary["geocell_id"].iloc[np.argmin(dists)]


GEOCELL_MAPPING_PATH = "data/geocells/image_geocell_mapping.csv"  # image_id, geocell_id, country

if os.path.exists(WORKING_GEOCELLS_PATH) and os.path.exists(GEOCELL_MAPPING_PATH):
    # a previously built (or migrated-in) scheme -- skip rebuilding entirely
    print("Loading cached geocells + image->geocell mapping")
    geocell_summary = pd.read_csv(WORKING_GEOCELLS_PATH)
    mapping = pd.read_csv(GEOCELL_MAPPING_PATH)[["image_id", "geocell_id", "country"]]
    images_df = images_df.merge(mapping, on="image_id")
elif len(images_df) >= CONFIG["min_points_for_geocells"]:
    print(f"{len(images_df)} points -- regenerating geocells from admin boundaries")
    assignments, geocell_summary = build_geocells(
        images_df, ADMIN0_PATH, ADMIN1_PATH,
        CONFIG["min_points_for_geocell_merge"], CONFIG["max_points_per_geocell"],
    )
    geocell_summary.to_csv(WORKING_GEOCELLS_PATH, index=False)
    assignments.to_csv(GEOCELL_MAPPING_PATH, index=False)
    images_df = images_df.merge(assignments, on="image_id")
else:
    print(f"Only {len(images_df)} points -- below the {CONFIG['min_points_for_geocells']} needed "
          f"to rebuild geocells reliably. Falling back to reference/geocell_summary.csv")
    geocell_summary = pd.read_csv(REFERENCE_GEOCELLS_PATH)
    geocell_summary.to_csv(WORKING_GEOCELLS_PATH, index=False)
    images_df["geocell_id"] = images_df.apply(
        lambda r: assign_to_nearest_geocell(r["latitude"], r["longitude"], geocell_summary), axis=1
    )
    images_df = images_df.merge(geocell_summary[["geocell_id", "country"]], on="geocell_id", how="left")
    images_df[["image_id", "geocell_id", "country"]].to_csv(GEOCELL_MAPPING_PATH, index=False)

geocell_summary = geocell_summary.sort_values("geocell_id").reset_index(drop=True)
CONFIG["num_geocells"] = len(geocell_summary)
print(f"Using {CONFIG['num_geocells']} geocells")

# finalize train/val dataframes now that geocell_id + country are attached
train_df = images_df[images_df["split"] == "train"].reset_index(drop=True)
val_df = images_df[images_df["split"] == "val"].reset_index(drop=True)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")


## 6. Embedding extraction

Runs every image through the frozen CLIP encoder once and caches the result —
retraining the head never needs to re-run the backbone.


In [ ]:
EMBEDDINGS_PATH = "data/embeddings/clip_embeddings.npz"

# Ensure the backbone is loaded even if Section 3 was skipped (e.g. no
# reference/ checkpoint exists yet) -- a full retrain shouldn't depend on
# the demo section having run successfully.
if "clip_model" not in dir():
    clip_model, clip_processor = load_clip_backbone(CONFIG["backbone"], DEVICE)

if os.path.exists(EMBEDDINGS_PATH):
    print("Loading cached embeddings")
    cached = np.load(EMBEDDINGS_PATH, allow_pickle=True)
    embedding_lookup = dict(zip(cached["image_ids"], cached["embeddings"]))
else:
    print(f"Extracting CLIP embeddings for {len(images_df)} images (this can take a while on CPU)")
    embedding_lookup = {}
    for _, row in images_df.iterrows():
        img_path = os.path.join(DATA_PATH, row["image_id"])
        try:
            embedding_lookup[row["image_id"]] = get_clip_embedding(img_path, clip_model, clip_processor, DEVICE)
        except FileNotFoundError:
            print(f"  missing image, skipping: {img_path}")

    if not embedding_lookup:
        raise RuntimeError(
            f"No images could be loaded from {DATA_PATH} -- check DATA_PATH and LABELS_CSV before continuing."
        )

    np.savez(
        EMBEDDINGS_PATH,
        image_ids=np.array(list(embedding_lookup.keys())),
        embeddings=np.array(list(embedding_lookup.values())),
    )
    print(f"Saved embeddings to {EMBEDDINGS_PATH}")

print(f"{len(embedding_lookup)} embeddings available")


## 7. Head training

The four-term loss: haversine-smoothed geocell classification, expected
haversine distance, a soft-threshold calibration loss (confidently-wrong is
penalized harder than honestly-unsure), and a soft country-match term (reward
probability mass placed on geocells in the correct country). `radius_a` /
`radius_b` are learned here, jointly with the classifier, via gradient descent
— they're what turns the model's own probability spread into a radius in km.


In [ ]:
from torch.utils.data import Dataset, DataLoader


class GeoDataset(Dataset):
    def __init__(self, df, embedding_lookup):
        self.df = df[df["image_id"].isin(embedding_lookup.keys())].reset_index(drop=True)
        self.embedding_lookup = embedding_lookup

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        embedding = torch.tensor(self.embedding_lookup[row["image_id"]], dtype=torch.float32)
        return {
            "embedding": embedding,
            "geocell_id": torch.tensor(int(row["geocell_id"]), dtype=torch.long),
            "lat": torch.tensor(row["latitude"], dtype=torch.float32),
            "lon": torch.tensor(row["longitude"], dtype=torch.float32),
            "country": row["country"],
        }


train_dataset = GeoDataset(train_df, embedding_lookup)
val_dataset = GeoDataset(val_df, embedding_lookup)

BATCH_SIZE = min(32, max(1, len(train_dataset)))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")
if len(train_dataset) < 500:
    print(f"Demo mode: training on {len(train_dataset)} images -- this will not reproduce "
          f"the reported results, see README for the full dataset.")

geocell_centroids = torch.tensor(
    geocell_summary[["centroid_lat", "centroid_lon"]].values, dtype=torch.float32, device=DEVICE
)
GEOCELL_COUNTRY = geocell_summary["country"].astype(str).values


def geocell_distance_matrix(centroids):
    n = centroids.shape[0]
    lat = centroids[:, 0].unsqueeze(1).expand(n, n)
    lon = centroids[:, 1].unsqueeze(1).expand(n, n)
    return haversine_torch(lat, lon, lat.T, lon.T)


# Full pairwise geocell-geocell distance matrix -- rebuilt from centroids on
# every run rather than saved to reference/ or data/. It's a deterministic
# function of geocell_summary's centroids (O(num_geocells^2), a few million
# entries at ~2005 cells -- negligible), so caching it on disk would only
# risk it drifting out of sync with whichever geocell_summary.csv is loaded.
CELL_DIST_MATRIX = geocell_distance_matrix(geocell_centroids)


def smoothed_classification_loss(logits, target_ids, tau=100.0):
    dists = CELL_DIST_MATRIX[target_ids]
    soft_targets = torch.softmax(-dists / tau, dim=-1)
    log_probs = F.log_softmax(logits, dim=-1)
    return -(soft_targets * log_probs).sum(dim=-1).mean()


def expected_distance_loss(logits, true_lat, true_lon):
    probs = F.softmax(logits, dim=-1)
    cell_lat = geocell_centroids[:, 0].unsqueeze(0).expand(true_lat.shape[0], -1)
    cell_lon = geocell_centroids[:, 1].unsqueeze(0).expand(true_lat.shape[0], -1)
    dists = haversine_torch(
        true_lat.unsqueeze(1).expand_as(cell_lat), true_lon.unsqueeze(1).expand_as(cell_lon),
        cell_lat, cell_lon,
    )
    return (probs * dists).sum(dim=-1).mean()


def calibration_loss(pred_radius, actual_error, T=50.0, S=500.0):
    covered = torch.sigmoid((pred_radius - actual_error) / T)
    tightness_penalty = pred_radius / S
    return (-torch.log(covered + 1e-6) + tightness_penalty).mean()


def country_bonus_penalty_loss(logits, true_country):
    probs = F.softmax(logits, dim=-1)
    match_mask = (GEOCELL_COUNTRY[None, :] == np.array(true_country)[:, None])
    match_mask_t = torch.tensor(match_mask, dtype=torch.float32, device=probs.device)
    prob_correct_country = (probs * match_mask_t).sum(dim=-1)
    return (1.0 - prob_correct_country).mean()


head = Head(embedding_dim=CONFIG["embedding_dim"], num_geocells=CONFIG["num_geocells"]).to(DEVICE)
optimizer = torch.optim.Adam(head.parameters(), lr=1e-3)

EPOCHS = 100
for epoch in range(EPOCHS):
    head.train()
    total_loss = 0.0
    for batch in train_loader:
        embeddings = batch["embedding"].to(DEVICE)
        geocell_ids = batch["geocell_id"].to(DEVICE)
        true_lat = batch["lat"].to(DEVICE)
        true_lon = batch["lon"].to(DEVICE)
        true_country = batch["country"]

        logits = head(embeddings)
        probs = F.softmax(logits, dim=-1)

        best_cell = torch.argmax(probs, dim=-1)
        pred_lat = geocell_centroids[best_cell, 0]
        pred_lon = geocell_centroids[best_cell, 1]
        actual_error = haversine_torch(pred_lat, pred_lon, true_lat, true_lon)

        spread = (probs * CELL_DIST_MATRIX[best_cell]).sum(dim=-1)
        pred_radius = radius_from_spread(spread, head.radius_a, head.radius_b)

        loss = (
            smoothed_classification_loss(logits, geocell_ids)
            + expected_distance_loss(logits, true_lat, true_lon)
            + calibration_loss(pred_radius, actual_error)
            + country_bonus_penalty_loss(logits, true_country)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{EPOCHS} -- loss: {total_loss / len(train_loader):.3f}")


## 8. Evaluation

Runs the trained head on the val set and reports median error, coverage
(fraction of true points falling inside the model's own stated radius), and
country accuracy. The radius reported here is used as-is — straight from
`radius_a`/`radius_b` — with no post-hoc scaling applied.


In [ ]:
head.eval()
records = []
with torch.no_grad():
    for batch in val_loader:
        embeddings = batch["embedding"].to(DEVICE)
        true_lat = batch["lat"].to(DEVICE)
        true_lon = batch["lon"].to(DEVICE)
        true_country = batch["country"]

        logits = head(embeddings)
        probs = F.softmax(logits, dim=-1)
        best_cell = torch.argmax(probs, dim=-1)
        pred_lat = geocell_centroids[best_cell, 0]
        pred_lon = geocell_centroids[best_cell, 1]
        spread = (probs * CELL_DIST_MATRIX[best_cell]).sum(dim=-1)
        radius = radius_from_spread(spread, head.radius_a, head.radius_b)
        error = haversine_torch(pred_lat, pred_lon, true_lat, true_lon)

        pred_country = GEOCELL_COUNTRY[best_cell.cpu().numpy()]

        for i in range(len(true_lat)):
            records.append({
                "pred_lat": pred_lat[i].item(),
                "pred_lon": pred_lon[i].item(),
                "true_lat": true_lat[i].item(),
                "true_lon": true_lon[i].item(),
                "radius": radius[i].item(),
                "error_km": error[i].item(),
                "pred_country": pred_country[i],
                "true_country": true_country[i],
            })

val_predictions = pd.DataFrame(records)

median_error = val_predictions["error_km"].median()
print(f"Median haversine error: {median_error:.1f} km")

country_accuracy = (val_predictions["pred_country"] == val_predictions["true_country"]).mean()
print(f"Country accuracy: {country_accuracy:.1%}")

coverage = (val_predictions["error_km"] <= val_predictions["radius"]).mean()
print(f"Coverage (error <= stated radius): {coverage:.1%}")

val_predictions.to_csv("data/predictions/val_predictions.csv", index=False)
print("Saved val_predictions.csv")


## 9. Visualization

A sample of 20 val points (predicted point + radius circle vs. actual point),
zoomed to the region they fall in, on a world map background, with labeled
longitude/latitude axes and a legend explaining the red circle. Saved to
`assets/`. If the world map background can't load, this prints why instead
of silently showing a blank background.


In [ ]:
import matplotlib.pyplot as plt


def plot_predictions_map(preds, n_samples=20, seed=SEED, save_path="assets/predictions_map.png"):
    sample = preds.sample(n=min(n_samples, len(preds)), random_state=seed)

    fig, ax = plt.subplots(figsize=(12, 8))

    # Zoom to the sampled points' region (with padding) rather than the whole
    # globe -- a world-scale view makes individual prediction errors (often
    # under a few hundred km) invisible regardless of any background.
    pad = 5.0  # degrees of padding around the points
    lon_min = min(sample["true_lon"].min(), sample["pred_lon"].min()) - pad
    lon_max = max(sample["true_lon"].max(), sample["pred_lon"].max()) + pad
    lat_min = min(sample["true_lat"].min(), sample["pred_lat"].min()) - pad
    lat_max = max(sample["true_lat"].max(), sample["pred_lat"].max()) + pad

    try:
        world = gpd.read_file(ADMIN0_PATH)
        world.plot(ax=ax, color="#e8e4d8", edgecolor="#888888", linewidth=0.5, zorder=1)
    except Exception as e:
        # Printed rather than silently ignored -- if the map is missing,
        # this tells you why instead of leaving a blank background unexplained.
        print(f"Could not load world map background from {ADMIN0_PATH} ({e}); plotting points without it.")

    for _, row in sample.iterrows():
        ax.plot(row["true_lon"], row["true_lat"], "go", markersize=7, zorder=5)
        ax.plot(row["pred_lon"], row["pred_lat"], "rx", markersize=8, mew=2, zorder=5)
        ax.plot([row["true_lon"], row["pred_lon"]], [row["true_lat"], row["pred_lat"]],
                color="gray", linewidth=0.5, alpha=0.5, zorder=4)
        radius_deg = row["radius"] / 111.0  # rough km-to-degrees for display only
        circle = plt.Circle((row["pred_lon"], row["pred_lat"]), radius_deg,
                             color="red", fill=False, alpha=0.5, linewidth=1.2, zorder=3)
        ax.add_patch(circle)

    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)
    ax.set_aspect(1.0)  # roughly equal degrees on both axes
    ax.set_xlabel("Longitude (degrees)")
    ax.set_ylabel("Latitude (degrees)")
    ax.grid(True, linestyle=":", alpha=0.4)

    ax.plot([], [], "go", label="True location")
    ax.plot([], [], "rx", label="Predicted location")
    ax.plot([], [], color="red", linewidth=1.2, label="Predicted radius (red circle)")
    ax.legend(loc="best")
    ax.set_title(f"Predicted vs. actual location ({len(sample)} sampled val points)")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


plot_predictions_map(val_predictions)


## 10. Save checkpoint

Saves the trained head and its config into `data/checkpoints/` — the working
copy, never `reference/`. Promoting a run to `reference/` (once you're happy
with it) is a manual step outside this notebook.


In [ ]:
CHECKPOINT_PATH = "data/checkpoints/head.pt"
CONFIG_PATH = "data/checkpoints/config.json"

torch.save(head.state_dict(), CHECKPOINT_PATH)
with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Saved trained head to {CHECKPOINT_PATH}")
print(f"Saved config to {CONFIG_PATH}")
print(json.dumps(CONFIG, indent=2))


## 11. Run inference on your own photos

Unlike Quick Inference (Section 3), this doesn't need ground-truth
coordinates — it's for the real use case: a folder of photos with unknown
locations, predicted in one pass. Independent of everything above it; it can
be run right after Section 1 and 2 without touching Sections 4–10 at all.


In [ ]:
CUSTOM_IMAGES_DIR = "data/custom_images"           # <-- drop your own photos in here
CUSTOM_OUTPUT_CSV = "data/predictions/custom_predictions.csv"

# Which trained model to use: reference/ (our validated checkpoint) by
# default, or swap to "data/checkpoints/" to use one you just retrained above.
CUSTOM_HEAD_PATH = "reference/head.pt"
CUSTOM_CONFIG_PATH = "reference/config.json"
CUSTOM_GEOCELLS_PATH = "reference/geocell_summary.csv"

image_files = sorted([
    f for f in os.listdir(CUSTOM_IMAGES_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

if not image_files:
    print(f"No images found in {CUSTOM_IMAGES_DIR}/ -- add some photos there and re-run this cell.")
else:
    print(f"Found {len(image_files)} images in {CUSTOM_IMAGES_DIR}")

    # Reloaded here (not reused from Section 3) so this cell runs standalone.
    custom_clip_model, custom_clip_processor = load_clip_backbone(CONFIG["backbone"], DEVICE)

    records = []
    for fname in image_files:
        pred_lat, pred_lon, pred_radius_km = predict(
            image_path=os.path.join(CUSTOM_IMAGES_DIR, fname),
            head_weights_path=CUSTOM_HEAD_PATH,
            config_path=CUSTOM_CONFIG_PATH,
            geocell_csv_path=CUSTOM_GEOCELLS_PATH,
            clip_model=custom_clip_model,
            clip_processor=custom_clip_processor,
            device=DEVICE,
        )
        records.append({
            "image_id": fname,
            "pred_lat": pred_lat,
            "pred_lon": pred_lon,
            "pred_radius_km": pred_radius_km,
        })
        print(f"{fname:40s} -> ({pred_lat:.4f}, {pred_lon:.4f})  +/- {pred_radius_km:.1f} km")

    os.makedirs(os.path.dirname(CUSTOM_OUTPUT_CSV), exist_ok=True)
    pd.DataFrame(records).to_csv(CUSTOM_OUTPUT_CSV, index=False)
    print(f"Saved {len(records)} predictions to {CUSTOM_OUTPUT_CSV}")
